# Run FashionAI on Google Colab\n
\n
This notebook runs the static frontend plus the FastAPI virtual try-on server in Colab.\n
\n
Before running: upload this project to Colab, mount Google Drive, or clone it from GitHub.

## 1. Get the project files\n
\n
Use **one** option below. For GitHub, set `REPO_URL`. For Drive/ZIP upload, adjust `PROJECT_DIR` after extracting.

In [ ]:
# Option A: clone from GitHub\n
REPO_URL = ''  # Example: 'https://github.com/your-user/AI-Multi-Model-Fashion-Assistant.git'\n
PROJECT_DIR = '/content/AI-Multi-Model-Fashion-Assistant'\n
\n
if REPO_URL:\n
    !rm -rf "$PROJECT_DIR"\n
    !git clone "$REPO_URL" "$PROJECT_DIR"\n
else:\n
    print('Set REPO_URL above, or upload/extract your project and set PROJECT_DIR manually.')

In [ ]:
# Option B: upload a ZIP manually, then run this cell.\n
# from google.colab import files\n
# uploaded = files.upload()\n
# ZIP_NAME = next(iter(uploaded))\n
# !rm -rf /content/fashionai-upload\n
# !mkdir -p /content/fashionai-upload\n
# !unzip -q "$ZIP_NAME" -d /content/fashionai-upload\n
# PROJECT_DIR = '/content/fashionai-upload/AI Multi-Model Fashion Assistant'\n
\n
import os\n
print('PROJECT_DIR =', PROJECT_DIR)\n
print('Exists:', os.path.exists(PROJECT_DIR))

## 2. Install dependencies

In [ ]:
%cd "$PROJECT_DIR/FastAPI-Server"\n
!pip install -q -r requirements.txt pyngrok\n
!python - <<'PY'\n
from pathlib import Path\n
for name in ['uploads', 'outputs', 'logs', 'models']:\n
    Path(name).mkdir(exist_ok=True)\n
print('FastAPI dependencies installed.')\n
PY

## 3. Start public tunnels\n
\n
Optional but recommended: add your ngrok auth token. Without it, ngrok may rate-limit or fail.

In [ ]:
NGROK_AUTH_TOKEN = ''  # Paste token from https://dashboard.ngrok.com/get-started/your-authtoken\n
\n
from pyngrok import ngrok, conf\n
if NGROK_AUTH_TOKEN:\n
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)\n
\n
ngrok.kill()\n
api_tunnel = ngrok.connect('127.0.0.1:8000', 'http')\n
frontend_tunnel = ngrok.connect('127.0.0.1:5173', 'http')\n
API_URL = api_tunnel.public_url\n
FRONTEND_URL = frontend_tunnel.public_url\n
print('FastAPI URL:', API_URL)\n
print('Frontend URL:', FRONTEND_URL)

## 4. Configure the frontend to call FastAPI directly

In [ ]:
from pathlib import Path\n
\n
html_path = Path(PROJECT_DIR) / 'Frontend' / 'virtual-tryon.html'\n
html = html_path.read_text(encoding='utf-8')\n
config_script = f'<script>window.FASHIONAI_TRYON_ENDPOINT = "{API_URL}/tryon";</script>'\n
\n
if 'window.FASHIONAI_TRYON_ENDPOINT' not in html:\n
    html = html.replace('</head>', f'    {config_script}\n</head>')\n
else:\n
    import re\n
    html = re.sub(r'<script>window\\.FASHIONAI_TRYON_ENDPOINT = .*?</script>', config_script, html)\n
\n
html_path.write_text(html, encoding='utf-8')\n
print('Configured virtual-tryon.html to call:', API_URL + '/tryon')

## 5. Start FastAPI and the frontend

In [ ]:
import os, subprocess, time\n
\n
# Make FastAPI accept the public frontend and return public output URLs.\n
server_env = os.environ.copy()\n
server_env['PUBLIC_BASE_URL'] = API_URL\n
server_env['CORS_ORIGINS'] = '[\"' + FRONTEND_URL + '\"]'\n
\n
api_proc = subprocess.Popen(\n
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],\n
    cwd=f'{PROJECT_DIR}/FastAPI-Server',\n
    env=server_env\n
)\n
\n
frontend_proc = subprocess.Popen(\n
    ['python', '-m', 'http.server', '5173', '--bind', '0.0.0.0'],\n
    cwd=f'{PROJECT_DIR}/Frontend'\n
)\n
\n
time.sleep(3)\n
print('FastAPI docs:', API_URL + '/docs')\n
print('Frontend home:', FRONTEND_URL + '/index.html')\n
print('Virtual try-on:', FRONTEND_URL + '/virtual-tryon.html')

## 6. Stop servers when finished

In [ ]:
# Run this when you are done.\n
# api_proc.terminate()\n
# frontend_proc.terminate()\n
# ngrok.kill()